In [1]:
from random import seed
from random import random

#initialize a network
def initialize_network(n_inputs, n_hidden, n_outputs):
  network = list()
  hidden_layer = [{'weights':[random() for i in range(n_inputs + 1)]} for i in range(n_hidden)]
  network.append(hidden_layer)
  output_layer = [{'weights':[random() for i in range(n_hidden + 1)]} for i in range(n_outputs)]
  network.append(output_layer)
  return network

In [2]:
seed(1)
network = initialize_network(2,1,2)
for layer in network:
  print(layer)

[{'weights': [0.13436424411240122, 0.8474337369372327, 0.763774618976614]}]
[{'weights': [0.2550690257394217, 0.49543508709194095]}, {'weights': [0.4494910647887381, 0.651592972722763]}]


In [3]:
from math import exp

#calculate neuron activation for an input
def activate(weights, inputs):
  activation = weights[-1]
  for i in range(len(weights)-1):
    activation += weights[i] * inputs[i]
  return activation

In [4]:
#transfer neuron activation with sigmoid activation
def transfer(activation):
  return 1.0 / (1.0 + exp(-activation))

In [5]:
#forward propagate input to a network output
def forward_propagate(network, row):
  inputs = row
  for layer in network:
    new_inputs = []
    for neuron in layer:
      activation = activate(neuron['weights'], inputs)
      neuron['output'] = transfer(activation)
      new_inputs.append(neuron['output'])
    inputs = new_inputs
  return inputs

In [6]:
#test forward propagation
network = [[{'weights': [0.13436424411240122, 0.8474337369372327, 0.763774618976614]}],
           [{'weights': [0.2550690257394217, 0.49543508709194095]}, {'weights': [0.4494910647887381,
                                                                                 0.651592972722763]}]]
row = [1, 0, None]
output = forward_propagate(network, row)
print(output)

[0.6629970129852887, 0.7253160725279748]


In [7]:
#calculate the derivation of a neuron output
def transfer_derivation(output):
  return output * (1.0 - output)

In [10]:
#backpropagate error and store in neurons
def backward_propagate_error(network, exprected):
  for i in reversed(range(len(network))):
    layer = network[i]
    errors = list()
    if i != len(network)-1:
      for j in range(len(layer)):
        error = 0.0
        for neuron in network[i+1]:
          error += (neuron['weights'][j] * neuron['delta'])
        errors.append(error)
    else:
      for j in range(len(layer)):
        neuron = layer[j]
        errors.append(neuron['output'] - expected[j])
    for j in range(len(layer)):
      neuron = layer[j]
      neuron['delta'] = errors[j] * transfer_derivation(neuron['output'])


In [11]:
#test backpropagation of error
network = [[{'output': 0.7105668883115941, 'weights': [0.13436424411240122, 0.8474337369372327, 0.763774618976614]}],
 [{'output': 0.6213859615555266, 'weights': [0.2550690257394217, 0.49543508709194095]}, {'output': 0.6573693455986976,
                                                                                         'weights': [0.4494910647887381, 0.651592972722763]}]]
expected = [0,1]
backward_propagate_error(network, expected)
for layer in network:
  print(layer)

[{'output': 0.7105668883115941, 'weights': [0.13436424411240122, 0.8474337369372327, 0.763774618976614], 'delta': 0.0005348048046610517}]
[{'output': 0.6213859615555266, 'weights': [0.2550690257394217, 0.49543508709194095], 'delta': 0.14619064683582808}, {'output': 0.6573693455986976, 'weights': [0.4494910647887381, 0.651592972722763], 'delta': -0.0771723774346327}]


In [12]:
#update network weights with error
def update_weights(network, row, l_rate):
  for i in range(len(network)):
    inputs = row[:-1]
    if i != 0:
      inputs = [neuron['output'] for neuron in network[i-1]]
    for neuron in network[i]:
      for j in range(len(inputs)):
        neuron['weights'][j] -= l_rate * neuron['delta'] * inputs[j]
      neuron['weights'][-1] -= l_rate * neuron['delta']

In [13]:
#train a network for a fixed number of epochs
def train_network(network, train, l_rate, n_epoch, n_outputs):
  for epoch in range(n_epoch):
    sum_error = 0
    for row in train:
      outputs = forward_propagate(network, row)
      expected = [0 for i in range(n_outputs)]
      expected[row[-1]] = 1
      sum_error += sum([(expected[i]-outputs[i])**2 for i in range(len(expected))])
      backward_propagate_error(network, expected)
      update_weights(network, row, l_rate)
    print('>epoch=%d, lrate=%.3f, error=%.3f' % (epoch, l_rate, sum_error))

In [14]:
#test training backprop algo
seed(1)
dataset = [[2.7810836,2.550537003,0],
           [1.465489372,2.362125076,0],
           [3.396561688,4.400293529,0],
           [1.38807019,1.850220317,0],
           [3.06407232,3.005305973,0],
           [7.627531214,2.759262235,1],
           [5.332441248,2.088626775,1],
           [6.922596716,1.77106367,1],
           [8.675418651,-0.242068655,1],
           [7.673756466,3.508563011,1]]
n_inputs = len(dataset[0]) - 1
n_outputs = len(set([row[-1] for row in dataset]))
network = initialize_network(n_inputs, 2, n_outputs)
train_network(network, dataset, 0.5, 20, n_outputs)
for layer in network:
  print(layer)

>epoch=0, lrate=0.500, error=5.020
>epoch=1, lrate=0.500, error=6.113
>epoch=2, lrate=0.500, error=7.187
>epoch=3, lrate=0.500, error=7.702
>epoch=4, lrate=0.500, error=8.011
>epoch=5, lrate=0.500, error=8.223
>epoch=6, lrate=0.500, error=8.380
>epoch=7, lrate=0.500, error=8.502
>epoch=8, lrate=0.500, error=8.600
>epoch=9, lrate=0.500, error=8.681
>epoch=10, lrate=0.500, error=8.750
>epoch=11, lrate=0.500, error=8.809
>epoch=12, lrate=0.500, error=8.861
>epoch=13, lrate=0.500, error=8.907
>epoch=14, lrate=0.500, error=8.947
>epoch=15, lrate=0.500, error=8.984
>epoch=16, lrate=0.500, error=9.017
>epoch=17, lrate=0.500, error=9.047
>epoch=18, lrate=0.500, error=9.074
>epoch=19, lrate=0.500, error=9.099
[{'weights': [0.21551530884274753, 0.8629941601651636, 0.7774442688955359], 'output': 0.9957590363464847, 'delta': -1.1382149316530232e-05}, {'weights': [0.3643242126612962, 0.5534819664732921, 0.4790041848356982], 'output': 0.9946001089073966, 'delta': -1.9498211872966402e-05}]
[{'weights

In [15]:
#make a prediction with a network
def predict(network, row):
  outputs = forward_propagate(network, row)
  return outputs.index(max(outputs))

In [16]:
network =  [[{'weights': [-1.482313569067226, 1.8308790073202204, 1.078381922048799]}, {'weights': [0.23244990332399884, 0.3621998343835864, 0.40289821191094327]}],
 [{'weights': [2.5001872433501404, 0.7887233511355132, -1.1026649757805829]}, {'weights': [-2.429358576245497, 0.8357651839198697, 1.0699217181280656]}]]
for row in dataset:
  prediction = predict(network, row)
  print('Expected=%d, Got=%d' % (row[-1], prediction))

Expected=0, Got=0
Expected=0, Got=0
Expected=0, Got=0
Expected=0, Got=0
Expected=0, Got=0
Expected=1, Got=1
Expected=1, Got=1
Expected=1, Got=1
Expected=1, Got=1
Expected=1, Got=1
